# Etapa 3 — Modelagem Preditiva: SARIMAX, baselines e validação walk-forward (2024 → 2025)
### Projeto Aplicado IV — Previsão de Casos de Arboviroses e Demanda Hospitalar (Rio de Janeiro)

Este notebook dá continuidade à Etapa 2 e cobre:

1. Carga dos dados reais de **dengue, chikungunya e zika** (InfoDengue) e clima (Open-Meteo), com **alinhamento correto das semanas epidemiológicas**
2. Seleção das defasagens climáticas **sem vazamento de informação** (apenas dados de treino, defasagem ≥ horizonte)
3. Modelos candidatos: **ARIMAX com termos de Fourier** (sazonalidade anual), **SARIMAX com sazonalidade S = 52** e **LSTM** (rede neural recorrente)
4. Baselines: **ingênuo sazonal** (mesma semana do ano anterior) e **persistência** (último valor observado)
5. Validação **walk-forward em três anos (2023, 2024 e 2025)**, com origem móvel: para cada ano, o modelo é treinado apenas com os anos anteriores e prevê de 1 a 4 semanas à frente
6. **Intervalos de previsão de 80%** e sua cobertura empírica
7. **Avaliação operacional por limiar de alerta** (Normal / Atenção / Crítico): o modelo avisa antes de a série cruzar o limiar?
8. Comparativo **2025 previsto × 2025 observado** e exportação das previsões para a etapa de desagregação por Área de Planejamento

> **Importante:** este notebook **não usa dados sintéticos**. Se alguma API estiver indisponível, a execução é interrompida
> de propósito, para que nenhum resultado artificial seja reportado.

## 0. Configuração

In [ ]:
# pip install statsmodels   # descomente se necessário

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.stats import pearsonr
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")          # avisos de convergência do statsmodels poluem a saída
plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.max_columns", None)

# ---------------- Parâmetros do experimento ----------------
H = 4                          # horizonte máximo de previsão (semanas à frente)
ANOS_TESTE = [2023, 2024, 2025] # validação em vários anos (rolling origin); o último é o ano "principal"
ALPHA_IC = 0.20                # intervalos de previsão de 80%
QUANTIS_ALERTA = (0.75, 0.90)  # limiares provisórios Atenção/Crítico (quantis dos casos no treino de cada ano)
                               # → serão substituídos pelos limiares baseados em leitos (CNES) na Etapa 4
PERIODO = 365.25 / 7           # ≈ 52,18 semanas por ano (usado nos termos de Fourier)
K_FOURIER = 3                  # nº de pares seno/cosseno para a sazonalidade anual
MAX_LAG_CLIMA = 12             # maior defasagem climática testada (semanas)
RODAR_SARIMA_52 = True         # SARIMAX com S=52 é lento (×3 anos de teste); mude para False se travar a máquina
RODAR_LSTM = True              # requer PyTorch; se não estiver instalado, a LSTM é pulada (rode no Google Colab)
JANELA_LSTM = 26               # semanas de histórico usadas como entrada da LSTM


def inicio_ano_epi(ano):
    """Domingo de início da SE 01 do ano epidemiológico (semana que contém 4 de janeiro)."""
    jan4 = pd.Timestamp(year=ano, month=1, day=4)
    return jan4 - pd.Timedelta(days=(jan4.dayofweek + 1) % 7)

INICIO_TESTE = inicio_ano_epi(ANOS_TESTE[-1])   # 29/12/2024 = SE 01/2025
print({a: inicio_ano_epi(a).date() for a in ANOS_TESTE})

DOENCAS = ["dengue", "chikungunya", "zika"]          # consultadas na API (exibidas na Figura 1)
DOENCAS_ALVO = ["dengue", "chikungunya"]             # somadas no alvo "arboviroses" (zika sem dados após fev/2024)
ALVO = "arboviroses"           # "arboviroses" (soma de DOENCAS_ALVO) | "dengue" | "chikungunya"

## 1. Carga dos dados reais

### 1.1 InfoDengue (três arboviroses) e Open-Meteo
Mesmas funções da Etapa 2, sem o fallback sintético. A API do InfoDengue é consultada uma vez para cada doença.

In [ ]:
def buscar_infodengue(geocode=3304557, doenca="dengue", ey_start=2015, ey_end=2025, ew_start=1, ew_end=52):
    url = "https://info.dengue.mat.br/api/alertcity"
    params = {"geocode": geocode, "disease": doenca, "format": "json",
              "ew_start": ew_start, "ew_end": ew_end, "ey_start": ey_start, "ey_end": ey_end}
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    if pd.api.types.is_numeric_dtype(df["data_iniSE"]):
        df["data_iniSE"] = pd.to_datetime(df["data_iniSE"], unit="ms")
    else:
        df["data_iniSE"] = pd.to_datetime(df["data_iniSE"])
    return df.set_index("data_iniSE").sort_index()


def buscar_clima_openmeteo(lat=-22.9068, lon=-43.1729, data_ini="2014-10-01", data_fim="2025-12-31"):
    # início em out/2014 para que as defasagens climáticas já existam nas primeiras semanas de 2015
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": data_ini, "end_date": data_fim,
              "daily": "temperature_2m_mean,temperature_2m_max,precipitation_sum",
              "timezone": "America/Sao_Paulo"}
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["daily"])
    df["time"] = pd.to_datetime(df["time"])
    return df.set_index("time").sort_index()


dados_doencas = {d: buscar_infodengue(doenca=d) for d in DOENCAS}
df_clima = buscar_clima_openmeteo()

for d, df in dados_doencas.items():
    assert df is not None and len(df) > 0, f"InfoDengue sem retorno para {d} — não prosseguir com dados sintéticos"
    print(f"InfoDengue {d:12s}: {df.index.min().date()} → {df.index.max().date()} ({len(df)} semanas)")
assert len(dados_doencas["dengue"]) > 500, "Série de dengue incompleta"
assert df_clima is not None and len(df_clima) > 3000, "Open-Meteo indisponível — não prosseguir com dados sintéticos"
print("Open-Meteo:", df_clima.index.min().date(), "→", df_clima.index.max().date())

### 1.2 Alinhamento das semanas epidemiológicas (correção da Etapa 2)

O InfoDengue indexa cada semana epidemiológica pelo **domingo de início** (`data_iniSE`, domingo → sábado).
Na Etapa 2, o clima foi agregado com `resample("W-SUN")`, que agrupa **segunda → domingo** e rotula pelo domingo **final**.
Na junção, os casos de uma semana eram pareados com o clima da semana anterior, o que introduzia uma defasagem oculta de ~1 semana.

A correção abaixo agrega o clima de domingo a sábado (`W-SAT`) e rotula cada semana pelo domingo de início,
exatamente como o InfoDengue.

In [ ]:
clima_semanal = df_clima.resample("W-SAT").agg(
    temp_media=("temperature_2m_mean", "mean"),
    temp_max=("temperature_2m_max", "mean"),
    precip_mm=("precipitation_sum", "sum"),
)
clima_semanal.index = clima_semanal.index - pd.Timedelta(days=6)   # rótulo = domingo de início da SE

# Uma coluna por doença, no mesmo índice semanal
idx_semanas = pd.date_range("2015-01-04", max(df.index.max() for df in dados_doencas.values()), freq="W-SUN")
casos_doencas = pd.DataFrame({d: df["casos_est"].astype(float).reindex(idx_semanas)
                              for d, df in dados_doencas.items()})

# Semanas ausentes:
#   - dentro do período coberto pela doença → interpolação
#   - antes do 1º registro → 0
#   - DEPOIS do último registro → permanecem ausentes (NaN): falta de dado não é ausência de casos
print("Semanas ausentes por doença (antes do tratamento):")
print(casos_doencas.isna().sum().to_string())
for d in DOENCAS:
    s_d = casos_doencas[d]
    primeiro, ultimo = s_d.first_valid_index(), s_d.last_valid_index()
    s_d = s_d.interpolate(method="time", limit_area="inside")
    s_d[s_d.index < primeiro] = 0
    casos_doencas[d] = s_d
    if ultimo < idx_semanas[-1]:
        print(f"[atenção] {d}: sem dados após {ultimo.date()} — excluída do alvo se não estiver completa")

incompletas = [d for d in DOENCAS_ALVO if casos_doencas[d].isna().any()]
assert not incompletas, f"Doenças do alvo com semanas sem dado: {incompletas}"
casos_doencas["arboviroses"] = casos_doencas[DOENCAS_ALVO].sum(axis=1)
casos = casos_doencas[ALVO].rename("casos")
print(f"\nAlvo da modelagem: {ALVO}")

# Checagem: os dois índices devem coincidir (domingos)
assert (casos.index.dayofweek == 6).all() and (clima_semanal.index.dayofweek == 6).all()
print("Índices alinhados por domingo de início da SE ✔")

### 1.3 Variável-alvo

O alvo padrão é a **soma semanal dos casos estimados de dengue e chikungunya** (`ALVO = "arboviroses"`).
A **zika foi excluída do alvo** porque a API do InfoDengue não retorna dados de zika para o Rio de Janeiro após fevereiro de 2024:
preencher esse período com zero criaria uma queda artificial justamente no ano de teste (2025).
No período em que as três séries existem, a zika representa uma fração pequena do total (ver participação abaixo).
Para modelar uma doença isoladamente, basta alterar `ALVO` na célula de configuração. A Figura 1 mostra a composição da série por doença.

Usamos `y = log(1 + casos)`. O λ ótimo de Box-Cox estimado na Etapa 2 foi ≈ −0,02, praticamente a transformação logarítmica,
que tem a vantagem de ser facilmente invertida (`exp(y) − 1`) e interpretada.

In [ ]:
y = np.log1p(casos).rename("y")
treino_mask = y.index < INICIO_TESTE
print(f"Treino: {y.index[treino_mask].min().date()} → {y.index[treino_mask].max().date()} ({treino_mask.sum()} semanas)")
print(f"Teste:  {y.index[~treino_mask].min().date()} → {y.index[~treino_mask].max().date()} ({(~treino_mask).sum()} semanas)")

fig, ax = plt.subplots()
ax.stackplot(casos_doencas.index, *[casos_doencas[d].fillna(0) for d in DOENCAS], labels=DOENCAS, alpha=0.85)
ax.axvline(pd.Timestamp(INICIO_TESTE), color="red", ls="--", label="Início do teste (SE 01/2025)")
ax.set_title("Figura 1 — Casos estimados de arboviroses por semana epidemiológica, Rio de Janeiro (2015–2025)")
ax.set_ylabel("Casos estimados"); ax.legend(loc="upper left"); plt.show()

# Participação de cada doença no total do período coberto por todas (até o último registro da zika)
fim_comum = min(casos_doencas[d].last_valid_index() for d in DOENCAS)
periodo = casos_doencas.loc[:fim_comum, DOENCAS]
part = (periodo.sum() / periodo.sum().sum() * 100).round(1)
print(f"Participação no total de casos, 2015 → {fim_comum.date()} (%):")
print(part.to_string())

## 2. Estratégia de validação em vários anos (origem móvel)

Avaliar só 2025 é frágil: foi um ano de baixa transmissão. Por isso, o mesmo procedimento é repetido para **2023, 2024 e 2025**.
Para cada ano de teste, **tudo** é recalculado usando apenas as semanas anteriores ao seu início, evitando vazamento de informação:

1. escolha das defasagens climáticas (correlação no treino, defasagem ≥ H)
2. padronização das variáveis climáticas (média e desvio do treino)
3. escolha da ordem do ARIMAX (AIC no treino)
4. estimação dos parâmetros e previsões *walk-forward* semanais de 1 a H semanas à frente, com intervalos de previsão de 80%
5. limiares provisórios de alerta (quantis dos casos no treino)

Durante o ano de teste, os parâmetros ficam fixos e o estado do modelo é atualizado semana a semana com o dado observado
(`append(..., refit=False)`), simulando o uso real.

### 2.1 Funções auxiliares

In [ ]:
clima_alinhado = clima_semanal.reindex(pd.date_range(clima_semanal.index.min(), y.index.max(), freq="W-SUN"))


def correlacao_por_defasagem(alvo, preditor, max_lag=MAX_LAG_CLIMA):
    res = {}
    for lag in range(0, max_lag + 1):
        d = pd.concat([alvo, preditor.shift(lag)], axis=1).dropna()
        res[lag] = pearsonr(d.iloc[:, 0], d.iloc[:, 1])[0]
    return pd.Series(res)


def termos_fourier(index, periodo=PERIODO, K=K_FOURIER, origem=pd.Timestamp("2015-01-04")):
    t = ((index - origem).days / 7).values
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(2 * np.pi * k * t / periodo)
        cols[f"cos{k}"] = np.cos(2 * np.pi * k * t / periodo)
    return pd.DataFrame(cols, index=index)


def preparar_fold(ano):
    """Monta tudo o que depende do treino de um ano de teste: máscaras, defasagens, exógenas e ordem do ARIMAX."""
    ini, fim = inicio_ano_epi(ano), inicio_ano_epi(ano + 1)
    treino = y.index < ini
    teste = (y.index >= ini) & (y.index < fim)

    correlacoes = {v: correlacao_por_defasagem(y[treino], clima_alinhado[v]) for v in ["temp_media", "precip_mm"]}
    lags = {v: int(s.loc[H:].abs().idxmax()) for v, s in correlacoes.items()}

    X_clima = pd.DataFrame({f"{v}_lag{L}": clima_alinhado[v].shift(L) for v, L in lags.items()}).reindex(y.index)
    X_clima = (X_clima - X_clima[treino].mean()) / X_clima[treino].std()
    X_fourier = termos_fourier(y.index)
    X_fourier_clima = pd.concat([X_fourier, X_clima], axis=1)
    assert not X_fourier_clima[treino | teste].isna().any().any(), "NaN nas exógenas"

    aic = []
    for p in range(0, 3):
        for q in range(0, 3):
            try:
                r = SARIMAX(y[treino], exog=X_fourier_clima[treino], order=(p, 0, q), trend="c").fit(disp=False)
                aic.append({"p": p, "q": q, "AIC": r.aic})
            except Exception as e:
                print(f"  ({ano}) p={p}, q={q} falhou: {e}")
    aic = pd.DataFrame(aic).sort_values("AIC").reset_index(drop=True)
    ordem = (int(aic.p[0]), 0, int(aic.q[0]))

    limiares = casos[treino].quantile(list(QUANTIS_ALERTA)).values
    return dict(ano=ano, treino=treino, teste=teste, correlacoes=correlacoes, lags=lags, ordem=ordem, aic=aic,
                X_clima=X_clima, X_fourier=X_fourier, X_fourier_clima=X_fourier_clima, limiares=limiares)


def origens_do_fold(f):
    """1ª origem = última semana do treino; depois, cada semana do teste exceto a última."""
    idx_teste = y.index[f["teste"]]
    return [y.index[f["treino"]][-1]] + list(idx_teste[:-1])


def walk_forward(f, X, ordem, ordem_sazonal=(0, 0, 0, 0), trend="c", nome="modelo"):
    treino, teste = f["treino"], f["teste"]
    res = SARIMAX(y[treino], exog=None if X is None else X[treino],
                  order=ordem, seasonal_order=ordem_sazonal, trend=trend).fit(disp=False)
    fim_teste = y.index[teste][-1]
    linhas = []
    for i, origem in enumerate(origens_do_fold(f)):
        if i > 0:  # incorpora a semana recém-observada
            res = res.append(y.loc[[origem]], exog=None if X is None else X.loc[[origem]], refit=False)
        pos = y.index.get_loc(origem)
        futuros = [d for d in y.index[pos + 1: pos + 1 + H] if d <= fim_teste]
        if not futuros:
            break
        fc = res.get_forecast(steps=len(futuros), exog=None if X is None else X.loc[futuros])
        media = np.clip(np.expm1(np.asarray(fc.predicted_mean)), 0, None)
        ic = np.clip(np.expm1(np.asarray(fc.conf_int(alpha=ALPHA_IC))), 0, None)   # quantis: invariantes à transformação
        for h, alvo in enumerate(futuros, start=1):
            linhas.append({"ano_teste": f["ano"], "modelo": nome, "origem": origem, "alvo": alvo, "h": h,
                           "previsto": media[h - 1], "ic_inf": ic[h - 1, 0], "ic_sup": ic[h - 1, 1]})
    return pd.DataFrame(linhas)


def baselines(f):
    fim_teste = y.index[f["teste"]][-1]
    linhas = []
    for origem in origens_do_fold(f):
        pos = casos.index.get_loc(origem)
        for h in range(1, H + 1):
            if pos + h >= len(casos) or casos.index[pos + h] > fim_teste:
                break
            alvo = casos.index[pos + h]
            base = {"ano_teste": f["ano"], "origem": origem, "alvo": alvo, "h": h, "ic_inf": np.nan, "ic_sup": np.nan}
            linhas.append({**base, "modelo": "Ingênuo sazonal", "previsto": casos.iloc[pos + h - 52]})
            linhas.append({**base, "modelo": "Persistência", "previsto": casos.iloc[pos]})
    return pd.DataFrame(linhas)

### 2.2 LSTM (aprendizado profundo)

A LSTM recebe as últimas `JANELA_LSTM` semanas de casos (em log, padronizados), dos termos de Fourier e do clima defasado,
e prevê **diretamente** as próximas H semanas (saída multi-horizonte). Para cada ano de teste, a rede é treinada uma única vez com os
dados anteriores a ele, usando as últimas 52 janelas do treino como validação para parada antecipada (*early stopping*).
Durante o ano de teste, os pesos ficam fixos e a rede só recebe as semanas observadas, como os demais modelos.

A LSTM não gera intervalos de previsão nesta versão, então ela não aparece na Tabela 3.

In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
except ImportError:
    TORCH_OK = False
    print("[aviso] PyTorch não instalado — a LSTM será pulada. Para incluí-la: pip install torch, ou rode no Google Colab.")

if TORCH_OK:
    class RedeLSTM(nn.Module):
        def __init__(self, n_variaveis, n_ocultas=32, horizonte=H):
            super().__init__()
            self.lstm = nn.LSTM(n_variaveis, n_ocultas, batch_first=True)
            self.dropout = nn.Dropout(0.2)
            self.saida = nn.Linear(n_ocultas, horizonte)

        def forward(self, x):
            saida, _ = self.lstm(x)
            return self.saida(self.dropout(saida[:, -1, :]))


def walk_forward_lstm(f, nome="LSTM", max_epocas=500, paciencia=30, semente=42):
    torch.manual_seed(semente)
    np.random.seed(semente)
    treino, teste = f["treino"], f["teste"]

    media_y, desvio_y = y[treino].mean(), y[treino].std()
    variaveis = pd.concat([((y - media_y) / desvio_y).rename("y"), f["X_fourier"], f["X_clima"]], axis=1)
    M = variaveis.values.astype("float32")
    alvo_pad = variaveis["y"].values.astype("float32")
    pos_fim_treino = int(np.where(treino)[0][-1])

    # janelas de treino: entrada termina em t; alvos t+1..t+H (todos dentro do treino)
    X_jan, Y_jan = [], []
    for t in range(JANELA_LSTM - 1, pos_fim_treino - H + 1):
        X_jan.append(M[t - JANELA_LSTM + 1: t + 1])
        Y_jan.append(alvo_pad[t + 1: t + 1 + H])
    X_jan, Y_jan = torch.tensor(np.array(X_jan)), torch.tensor(np.array(Y_jan))
    n_val = 52
    X_tr, Y_tr, X_val, Y_val = X_jan[:-n_val], Y_jan[:-n_val], X_jan[-n_val:], Y_jan[-n_val:]

    rede = RedeLSTM(M.shape[1])
    otimizador = torch.optim.Adam(rede.parameters(), lr=1e-3)
    perda_fn = nn.MSELoss()
    melhor_perda, melhor_estado, sem_melhora = np.inf, None, 0
    for epoca in range(max_epocas):
        rede.train()
        otimizador.zero_grad()
        perda = perda_fn(rede(X_tr), Y_tr)
        perda.backward()
        otimizador.step()
        rede.eval()
        with torch.no_grad():
            perda_val = perda_fn(rede(X_val), Y_val).item()
        if perda_val < melhor_perda - 1e-5:
            melhor_perda, sem_melhora = perda_val, 0
            melhor_estado = {k: v.clone() for k, v in rede.state_dict().items()}
        else:
            sem_melhora += 1
            if sem_melhora >= paciencia:
                break
    rede.load_state_dict(melhor_estado)
    rede.eval()

    fim_teste = y.index[teste][-1]
    linhas = []
    for origem in origens_do_fold(f):
        pos = y.index.get_loc(origem)
        futuros = [d for d in y.index[pos + 1: pos + 1 + H] if d <= fim_teste]
        if not futuros:
            break
        with torch.no_grad():
            saida = rede(torch.tensor(M[None, pos - JANELA_LSTM + 1: pos + 1])).numpy()[0]
        prev = np.clip(np.expm1(saida * desvio_y + media_y), 0, None)
        for h, alvo in enumerate(futuros, start=1):
            linhas.append({"ano_teste": f["ano"], "modelo": nome, "origem": origem, "alvo": alvo, "h": h,
                           "previsto": prev[h - 1], "ic_inf": np.nan, "ic_sup": np.nan})
    print(f"       LSTM: {epoca + 1} épocas, perda de validação {melhor_perda:.4f}")
    return pd.DataFrame(linhas)

### 2.3 Execução para cada ano de teste

Esta é a célula mais demorada do notebook (o SARIMAX com S = 52 e a LSTM são estimados uma vez por ano de teste).

In [ ]:
import time

folds, prev_lista = {}, []
for ano in ANOS_TESTE:
    t0 = time.time()
    f = preparar_fold(ano)
    folds[ano] = f
    print(f"[{ano}] treino até {y.index[f['treino']][-1].date()} | defasagens {f['lags']} | ordem ARIMAX {f['ordem']} "
          f"| limiares Atenção/Crítico = {f['limiares'].round(0)}")
    prev_lista.append(baselines(f))
    prev_lista.append(walk_forward(f, f["X_fourier_clima"], f["ordem"], nome="ARIMAX + Fourier + clima"))
    prev_lista.append(walk_forward(f, f["X_fourier"], f["ordem"], nome="ARIMAX + Fourier (sem clima)"))
    if RODAR_SARIMA_52:
        prev_lista.append(walk_forward(f, f["X_clima"], (1, 0, 1), ordem_sazonal=(1, 0, 0, 52),
                                       nome="SARIMAX(1,0,1)(1,0,0)52 + clima"))
    if RODAR_LSTM and TORCH_OK:
        prev_lista.append(walk_forward_lstm(f))
    print(f"       concluído em {time.time() - t0:.0f} s")

previsoes = pd.concat(prev_lista, ignore_index=True)
previsoes["observado"] = casos.reindex(previsoes["alvo"]).values
previsoes["erro_abs"] = (previsoes["previsto"] - previsoes["observado"]).abs()
previsoes.groupby(["ano_teste", "modelo"]).size().unstack(0)

### 2.4 Defasagens climáticas no ano principal (2025)

A Figura 2 mostra a correlação entre o clima defasado e os casos no treino do ano principal. A área cinza marca as defasagens
menores que H, que não podem ser usadas sem vazamento de informação.

In [ ]:
f = folds[ANOS_TESTE[-1]]
tab_corr = pd.DataFrame(f["correlacoes"]).round(3)
tab_corr.index.name = "defasagem (semanas)"

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (v, cor) in zip(axes, [("temp_media", "firebrick"), ("precip_mm", "steelblue")]):
    ax.bar(tab_corr.index, tab_corr[v], color=cor)
    ax.axvspan(-0.5, H - 0.5, color="grey", alpha=0.15, label=f"defasagens < H={H} (não usadas)")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{v} defasada × log(1+casos) — treino até {ANOS_TESTE[-1] - 1}")
    ax.set_xlabel("Defasagem (semanas)"); ax.legend()
plt.suptitle("Figura 2 — Correlação cruzada entre clima defasado e casos", y=1.03)
plt.tight_layout(); plt.show()
tab_corr

## 3. Resultados

### 3.1 Erro de previsão por horizonte

A Tabela 1 apresenta o MAE, o RMSE (em casos semanais) e o **WAPE** (erro absoluto total ÷ casos observados totais) de cada modelo
por horizonte, **agregando os três anos de teste**. A coluna **Ganho vs. persistência** indica a redução percentual do MAE em relação
ao baseline de persistência (repetir o último valor observado); valores positivos indicam que o modelo supera esse baseline.

> **Por que persistência como referência?** Em anos que sucedem epidemias (como 2025, após 2024), repetir o ano anterior gera erros
> muito altos, o que faria qualquer modelo parecer excelente. A persistência é um baseline mais exigente para horizontes curtos.
> O ingênuo sazonal é mantido nas tabelas apenas como referência.

In [ ]:
def metricas(g):
    e = g["previsto"] - g["observado"]
    return pd.Series({"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()),
                      "WAPE (%)": e.abs().sum() / g["observado"].sum() * 100, "n": len(g)})


def tabela_metricas(df, chaves):
    t = df.groupby(chaves + ["modelo"]).apply(metricas).reset_index()
    ref = t[t.modelo == "Persistência"].set_index(chaves)["MAE"]
    t["Ganho vs. persistência (%)"] = (1 - t["MAE"] / t.set_index(chaves).index.map(ref)) * 100
    t["n"] = t["n"].astype(int)
    return t.round({"MAE": 1, "RMSE": 1, "WAPE (%)": 1, "Ganho vs. persistência (%)": 1})


tabela1 = tabela_metricas(previsoes, ["h"])
print(f"Tabela 1 — Desempenho walk-forward por horizonte, anos de teste {ANOS_TESTE} agregados")
tabela1.sort_values(["h", "MAE"])

A Tabela 1b detalha o **ganho sobre a persistência por ano de teste**, para verificar se o resultado é consistente
ou depende de um ano específico.

In [ ]:
tabela1b = tabela_metricas(previsoes, ["ano_teste", "h"])
tabela1b_pivot = tabela1b.pivot_table(index=["modelo", "h"], columns="ano_teste", values="Ganho vs. persistência (%)")
print("Tabela 1b — Ganho vs. persistência (%) por ano de teste e horizonte")
tabela1b_pivot.drop(index="Persistência", level=0)

### 3.2 Intervalos de previsão (80%)

Para a gestão, uma faixa provável de casos é mais útil do que um número único. A Tabela 3 mostra a **cobertura empírica**
dos intervalos de 80% (o ideal é ficar próxima de 80%) e a **largura média** do intervalo, em casos.
Cobertura abaixo de 80% indica intervalos otimistas demais; muito acima, intervalos largos demais para serem úteis.

In [ ]:
com_ic = previsoes.dropna(subset=["ic_inf"]).copy()
com_ic["dentro"] = (com_ic["observado"] >= com_ic["ic_inf"]) & (com_ic["observado"] <= com_ic["ic_sup"])
com_ic["largura"] = com_ic["ic_sup"] - com_ic["ic_inf"]
tabela3 = (com_ic.groupby(["modelo", "h"])
           .agg(**{"Cobertura (%)": ("dentro", lambda s: s.mean() * 100),
                   "Largura média (casos)": ("largura", "mean")})
           .round(1).reset_index())
print(f"Tabela 3 — Cobertura dos intervalos de previsão de {int((1 - ALPHA_IC) * 100)}%, anos de teste agregados")
tabela3.pivot_table(index="modelo", columns="h", values=["Cobertura (%)", "Largura média (casos)"])

### 3.3 Avaliação operacional por limiar de alerta

A gestão de leitos depende menos do número exato de casos e mais de **saber com antecedência quando a demanda vai mudar de patamar**.
Cada semana é classificada em **Normal**, **Atenção** ou **Crítico** pelos limiares provisórios (quantis 75% e 90% dos casos no treino
de cada ano). Na Etapa 4, esses limiares serão substituídos por limiares calculados a partir da capacidade de leitos (CNES).

São avaliados:

- **Acerto do nível** (Normal / Atenção / Crítico) da semana prevista
- **Sensibilidade** para "Atenção ou acima": das semanas que de fato atingiram o limiar, quantas o modelo previu
- **Precisão**: dos alertas emitidos, quantos se confirmaram (1 − taxa de alarme falso)
- **Antecedência na entrada em alerta**: em cada semana em que a série cruza o limiar de Atenção, quantas semanas antes algum modelo já previa o cruzamento

In [ ]:
def nivel(valor, limiares):
    return int(np.select([valor >= limiares[1], valor >= limiares[0]], [2, 1], default=0))   # 0 Normal, 1 Atenção, 2 Crítico

previsoes["limiar_atencao"] = previsoes["ano_teste"].map({a: f["limiares"][0] for a, f in folds.items()})
previsoes["nivel_prev"] = [nivel(v, folds[a]["limiares"]) for v, a in zip(previsoes["previsto"], previsoes["ano_teste"])]
previsoes["nivel_obs"] = [nivel(v, folds[a]["limiares"]) for v, a in zip(previsoes["observado"], previsoes["ano_teste"])]


def avaliar_alerta(g):
    alerta_prev, alerta_obs = g["nivel_prev"] >= 1, g["nivel_obs"] >= 1
    vp = (alerta_prev & alerta_obs).sum()
    return pd.Series({
        "Acerto do nível (%)": (g["nivel_prev"] == g["nivel_obs"]).mean() * 100,
        "Sensibilidade (%)": vp / alerta_obs.sum() * 100 if alerta_obs.sum() else np.nan,
        "Precisão (%)": vp / alerta_prev.sum() * 100 if alerta_prev.sum() else np.nan,
        "Semanas em alerta (obs.)": int(alerta_obs.sum()),
    })

tabela4 = previsoes.groupby(["modelo", "h"]).apply(avaliar_alerta).round(1).reset_index()
print("Tabela 4 — Desempenho na classificação de alerta (Atenção ou acima), anos de teste agregados")
tabela4.pivot_table(index="modelo", columns="h", values=["Sensibilidade (%)", "Precisão (%)", "Acerto do nível (%)"])

A Tabela 5 mede a **antecedência na entrada em alerta**: para cada semana em que os casos observados cruzaram o limiar de Atenção
(vindo de uma semana abaixo dele), verifica-se a origem mais antiga, entre 1 e H semanas antes, cuja previsão já indicava o cruzamento.

In [ ]:
linhas = []
for ano, f in folds.items():
    L = f["limiares"][0]
    obs = casos[f["treino"] | f["teste"]]
    semanas_teste = y.index[f["teste"]]
    entradas = [t for t in semanas_teste if obs[t] >= L and obs.shift(1)[t] < L]
    for t in entradas:
        for m in previsoes.modelo.unique():
            p = previsoes[(previsoes.ano_teste == ano) & (previsoes.modelo == m) & (previsoes.alvo == t)]
            avisos = p[p["previsto"] >= L]
            linhas.append({"ano_teste": ano, "entrada em alerta": t.date(), "modelo": m,
                           "antecedência (semanas)": int(avisos["h"].max()) if len(avisos) else 0})

tab_entradas = pd.DataFrame(linhas)
if tab_entradas.empty:
    print("Nenhuma entrada em alerta nos anos de teste com os limiares atuais.")
    tabela5 = pd.DataFrame()
else:
    tabela5 = (tab_entradas.groupby("modelo")
               .agg(**{"Entradas em alerta": ("antecedência (semanas)", "size"),
                       "Avisadas com antecedência (%)": ("antecedência (semanas)", lambda s: (s > 0).mean() * 100),
                       "Antecedência média (semanas)": ("antecedência (semanas)", "mean")})
               .round(1).sort_values("Antecedência média (semanas)", ascending=False))
    print("Tabela 5 — Antecedência na entrada em alerta (limiar de Atenção), anos de teste agregados")
tabela5

### 3.4 Comparativo do ano principal: previsto × observado

A Figura 3 compara, para cada ano de teste, a série observada com as previsões feitas com H semanas de antecedência pelo melhor modelo,
incluindo o intervalo de previsão de 80% e os limiares de alerta.

In [ ]:
modelos_ml = [m for m in previsoes.modelo.unique() if m not in ("Ingênuo sazonal", "Persistência")]
melhor_modelo = tabela1[(tabela1.h == H) & tabela1.modelo.isin(modelos_ml)].sort_values("MAE").iloc[0]["modelo"]
print("Melhor modelo em h =", H, "→", melhor_modelo)

fig, axes = plt.subplots(len(ANOS_TESTE), 1, figsize=(13, 3.6 * len(ANOS_TESTE)))
for ax, ano in zip(np.atleast_1d(axes), ANOS_TESTE):
    f = folds[ano]
    obs = casos[f["teste"]]
    s = previsoes[(previsoes.ano_teste == ano) & (previsoes.modelo == melhor_modelo) & (previsoes.h == H)].set_index("alvo")
    ax.fill_between(s.index, s["ic_inf"], s["ic_sup"], alpha=0.2, label=f"IC {int((1 - ALPHA_IC) * 100)}%")
    ax.plot(obs.index, obs, color="black", lw=2, label="Observado")
    ax.plot(s.index, s["previsto"], ls="--", label=f"Previsto {H} semanas antes")
    for lim, nome, cor in zip(f["limiares"], ["Atenção", "Crítico"], ["orange", "red"]):
        ax.axhline(lim, color=cor, ls=":", lw=1, label=f"Limiar {nome}")
    ax.set_title(f"{ano}"); ax.set_ylabel("Casos/semana"); ax.legend(loc="upper right", fontsize=8)
plt.suptitle(f"Figura 3 — Observado × previsto com {H} semanas de antecedência ({melhor_modelo})", y=1.0)
plt.tight_layout(); plt.show()

### 3.5 Pico de cada ano de teste

A Tabela 2 compara a semana do pico observado com a semana do pico previsto com H semanas de antecedência, em cada ano de teste.

In [ ]:
linhas = []
for ano, f in folds.items():
    obs = casos[f["teste"]]
    pico = obs.idxmax()
    linhas.append({"ano_teste": ano, "modelo": "Observado", "semana do pico": pico.date(),
                   "casos no pico": round(obs.max()), "diferença (semanas)": 0})
    for m in previsoes.modelo.unique():
        s = previsoes[(previsoes.ano_teste == ano) & (previsoes.modelo == m) & (previsoes.h == H)].set_index("alvo")["previsto"]
        linhas.append({"ano_teste": ano, "modelo": f"{m} (h={H})", "semana do pico": s.idxmax().date(),
                       "casos no pico": round(s.max()), "diferença (semanas)": int((s.idxmax() - pico).days / 7)})
tabela2 = pd.DataFrame(linhas)
print(f"Tabela 2 — Pico observado × previsto com {H} semanas de antecedência")
tabela2

### 3.6 Exportação

As previsões (com intervalos e níveis de alerta) e as tabelas são salvas para a próxima etapa
(desagregação *top-down* por Área de Planejamento e camada de alerta hospitalar).

In [ ]:
previsoes.to_csv(f"previsoes_walkforward_{ALVO}.csv", index=False)
casos_doencas.assign(casos_alvo=casos).rename_axis("semana").to_csv(f"serie_semanal_{ALVO}.csv")   # usada na Etapa 4
for nome, t in {"tabela1_metricas": tabela1, "tabela1b_ganho_por_ano": tabela1b, "tabela2_pico": tabela2,
                "tabela3_intervalos": tabela3, "tabela4_alerta": tabela4, "tabela5_antecedencia": tabela5}.items():
    t.to_csv(f"{nome}_{ALVO}.csv", index=False)
print("Arquivos salvos com sufixo", ALVO)

## 4. Limitações a declarar

- **Soma de doenças com dinâmicas diferentes:** dengue e chikungunya têm sazonalidade e surtos próprios; somá-las favorece a gestão de leitos (demanda total), mas mistura sinais epidemiológicos. A modelagem por doença fica disponível via `ALVO`.
- **Zika fora do alvo:** a API do InfoDengue não retorna zika para o município após fev/2024; a doença aparece apenas na análise descritiva.
- **Limiares de alerta provisórios:** baseados em quantis históricos de casos, não ainda na capacidade de leitos; serão recalibrados na Etapa 4 e com profissionais do SUS-RJ.
- **Atraso de notificação:** as semanas mais recentes podem estar subnotificadas; `casos_est` do InfoDengue já aplica correção (*nowcasting*), mas resta incerteza.
- **Parâmetros fixos dentro de cada ano de teste:** os coeficientes são estimados no início de cada ano e não reestimados durante ele.
- **Intervalos de previsão paramétricos:** assumem erros aproximadamente normais na escala log; a cobertura empírica (Tabela 3) indica se essa hipótese é razoável.
- **LSTM sem intervalos e com treino único por ano:** a rede é sensível à semente aleatória e à arquitetura; os hiperparâmetros não foram otimizados.
- **Retransformação:** `exp(ŷ) − 1` estima a mediana, não a média, dos casos (viés de retransformação do log).